<a href="https://colab.research.google.com/github/tush7301/CAST_POC/blob/main/LlamaIndex_POC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!pip install -q groq
!pip install -q llama-index-llms-groq
!pip install -q llama-parse
!pip install -q llama-index-embeddings-huggingface
!pip install -q sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-readers-file 0.6.0 requires pypdf<7,>=6.1.3, which is not installed.
llama-index-readers-file 0.6.0 requires striprtf<0.0.27,>=0.0.26, which is not installed.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [30]:
import os

os.environ["LLAMA_CLOUD_API_KEY"] = ""
os.environ["GROQ_API_KEY"] = ""

In [31]:
import nest_asyncio
nest_asyncio.apply()

In [32]:
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
import os

llm = Groq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0,
)
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

print("Groq (Llama 3.3 70B) + HuggingFace embeddings configured")

Groq (Llama 3.3 70B) + HuggingFace embeddings configured


In [33]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

drive_pdf_dir = "/content/drive/MyDrive/research_papers"
papers = [str(p) for p in Path(drive_pdf_dir).glob("*.pdf")]

print(f"Found {len(papers)} PDFs:")
for p in papers:
    print(f"  - {Path(p).name}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 58 PDFs:
  - Australian Youth Mental Health and Climate Change.pdf
  - Integrated fire management as an adaptation and mi.pdf
  - Public Health Messaging for Wildfire Smoke Cast a.pdf
  - Reimagine fire science for the anthropocene.pdf
  - Bushfire Smoke and Childrens Health-Exploring a C.pdf
  - Social group connections support mental health fol.pdf
  - Efficacy of Communication Techniques and Health Ou.pdf
  - Wildfire smoke and health impacts a narrative rev.pdf
  - California and Oregon NICU Wildfire Disaster Prepa.pdf
  - Eco-Anxiety and Mental Health Correlates of Clima.pdf
  - A Systematic Review of the Impacts of Climate Chan.pdf
  - Child-focused climate change and health content in.pdf
  - Investigating the Health Impacts of Climate Change.pdf
  - Wildfires and Older Adults A Scoping Review of Im.pdf
  - Wildfire Environmental Risk and Deliber

In [34]:
index_dir = "/content/drive/MyDrive/research_papers/index"

if Path(index_dir).exists():
    print("Saved index found — will load instead of re-parsing")
    use_saved = True
else:
    print("No saved index found — will parse all papers")
    use_saved = False

No saved index found — will parse all papers


In [35]:
import warnings
warnings.filterwarnings("ignore")

from llama_index.core import VectorStoreIndex, StorageContext, load_index_from_storage
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_parse import LlamaParse

if use_saved:
    # Load from Drive — no re-parsing needed
    print("Loading saved index from Drive...")
    storage_context = StorageContext.from_defaults(persist_dir=index_dir)
    combined_index = load_index_from_storage(storage_context)
    print("Index loaded successfully")

else:
    # Parse all papers and build index
    all_nodes = []

    for paper in papers:
        name = Path(paper).stem
        print(f"\nParsing: {name}")
        try:
            parser = LlamaParse(result_type="text")
            json_objs = parser.get_json_result(paper)
            json_list = json_objs[0]["pages"]
            docs = [
                Document(
                    text=page["text"],
                    metadata={"source": name, "page_label": page["page"]}
                )
                for page in json_list
            ]
            nodes = SentenceSplitter(chunk_size=1024).get_nodes_from_documents(docs)
            all_nodes.extend(nodes)
            print(f"Added {len(nodes)} chunks")
        except Exception as e:
            print(f"Skipping {name}: {e}")

    print(f"\nBuilding index over {len(all_nodes)} total chunks...")
    combined_index = VectorStoreIndex(all_nodes)

    # Save to Drive for future sessions
    print("Saving index to Drive...")
    combined_index.storage_context.persist(persist_dir=index_dir)
    print(f"Index saved to {index_dir}")


Parsing: Australian Youth Mental Health and Climate Change
Started parsing the file under job_id 36d718f7-8968-4e65-a994-5a2e8ddf62be
Added 25 chunks

Parsing: Integrated fire management as an adaptation and mi
Started parsing the file under job_id 0598643d-04a1-4206-9122-4dd7d7274f84
Added 11 chunks

Parsing: Public Health Messaging for Wildfire Smoke Cast a
Started parsing the file under job_id 4f023046-3d95-44ae-a8e3-9e193f7eec59
Added 73 chunks

Parsing: Reimagine fire science for the anthropocene
Started parsing the file under job_id 0598643d-04a1-4206-9122-4dd7d7274f84
Added 11 chunks

Parsing: Bushfire Smoke and Childrens Health-Exploring a C
Started parsing the file under job_id 4f023046-3d95-44ae-a8e3-9e193f7eec59
Added 73 chunks

Parsing: Social group connections support mental health fol
Started parsing the file under job_id 72b7f3fd-aa16-4476-9c96-f4caa43d1b82
Added 10 chunks

Parsing: Efficacy of Communication Techniques and Health Ou
Started parsing the file under job_id

In [17]:
query_engine = combined_index.as_query_engine(
    similarity_top_k=10,
    verbose=True
)

print("Query engine ready")

Started parsing the file under job_id 74508bc3-c801-4989-acf9-e13bd97d83c4
✅ Added: Australian Youth Mental Health and Climate Change (25 chunks)
Started parsing the file under job_id 9585a0a6-04c0-4c2e-b9ba-4527df6ccc61
✅ Added: Integrated fire management as an adaptation and mi (11 chunks)
Started parsing the file under job_id 75444a8d-9eda-4c9c-bc83-2e706737c069
✅ Added: Public Health Messaging for Wildfire Smoke Cast a (73 chunks)
Started parsing the file under job_id 9585a0a6-04c0-4c2e-b9ba-4527df6ccc61
✅ Added: Reimagine fire science for the anthropocene (11 chunks)
Started parsing the file under job_id 75444a8d-9eda-4c9c-bc83-2e706737c069
✅ Added: Bushfire Smoke and Childrens Health-Exploring a C (73 chunks)
Started parsing the file under job_id 5eedabd0-3b51-4023-ad5b-393f59e7a940
✅ Added: Social group connections support mental health fol (10 chunks)
Started parsing the file under job_id 75444a8d-9eda-4c9c-bc83-2e706737c069
✅ Added: Efficacy of Communication Techniques and Hea

In [36]:
response =  query_engine.query(("What kinds of physical and mental health impacts are associated with wildfire exposure?"))
print(str(response))

Exposure to wildfire smoke has been linked to various physical and mental health issues. Respiratory problems, such as asthma and chronic obstructive pulmonary disease (COPD), can be exacerbated by the fine particulate matter and other pollutants present in wildfire smoke. This can lead to symptoms like coughing, wheezing, and shortness of breath.

In addition to respiratory issues, wildfire smoke exposure has also been associated with cardiovascular problems, including heart attacks, strokes, and arrhythmias. The particulate matter in smoke can cause inflammation in the blood vessels, leading to these cardiovascular issues.

Mental health impacts of wildfire exposure are also significant. The trauma and stress caused by wildfires can lead to anxiety, depression, and post-traumatic stress disorder (PTSD). The loss of homes and property, as well as the disruption of daily life, can contribute to these mental health issues.

Furthermore, exposure to wildfire smoke has been linked to neur

In [37]:
response = query_engine.query("How does wildfire smoke, especially PM2.5, affect respiratory and cardiovascular health?")
print(str(response))

Exposure to wildfire smoke, particularly fine particulate matter (PM2.5), poses significant health risks, particularly for respiratory and cardiovascular conditions. Studies have shown that wildfire smoke can lead to a 21% increased risk of developing dementia, more so than other air pollutants. Additionally, it may impact cognition, leading to lower test scores and reduced future income. Furthermore, exposure to wildfire smoke has been linked to serious health risks, including respiratory problems, hospital visits spiking during smoke events, and increased risk of cardiovascular disease.


In [38]:
response = query_engine.query("What kinds of mental health support might people need after wildfires?")
print(str(response))

After experiencing a traumatic event like a wildfire, individuals may require various forms of mental health support to cope with their emotions and reactions. Some potential types of support include:

1. Counseling or therapy to process their emotions and develop coping strategies.
2. Group support sessions to connect with others who have experienced similar trauma.
3. Access to crisis hotlines or online resources for immediate support.
4. Education on stress management techniques, such as mindfulness and relaxation methods.
5. Referrals to specialized mental health services, such as trauma-focused therapy or cognitive-behavioral therapy.
6. Support from loved ones, friends, or community members to help rebuild a sense of safety and security.
7. Opportunities for physical activity or exercise to release tension and improve mood.
8. Access to resources for managing anxiety, depression, or post-traumatic stress disorder (PTSD).
9. Support for children and families to address specific ne

In [39]:
response = query_engine.query("How should public health agencies communicate wildfire smoke risks to different communities?")
print(str(response))

Effective communication of wildfire smoke risks to different communities requires a multi-faceted approach that takes into account the unique needs and concerns of each group. Public health agencies should consider the following strategies:

1. **Clear and concise messaging**: Use simple, easy-to-understand language to convey the risks associated with wildfire smoke, including the potential health effects and recommended precautions.
2. **Cultural sensitivity**: Tailor messages to the cultural and linguistic backgrounds of the communities being served, using culturally relevant examples and terminology.
3. **Visual aids**: Utilize visual aids such as maps, graphics, and videos to help illustrate the risks and provide a clear understanding of the situation.
4. **Multiple channels**: Leverage various communication channels, including social media, local news outlets, community events, and traditional media, to reach a wide audience.
5. **Community engagement**: Engage with community lead

In [40]:
response = query_engine.query("What wildfire smoke communication barriers affect non-English-speaking families and other vulnerable groups?")
print(str(response))

Language barriers can hinder the effectiveness of wildfire smoke communication, particularly for non-English-speaking families. This can lead to a lack of understanding about the risks associated with wildfire smoke, making it challenging for them to take necessary precautions to protect themselves.

Additionally, cultural and linguistic differences can also create barriers to effective communication. For instance, some communities may have limited access to information in their native language, or they may have different cultural norms and values that influence their perception of risk and their willingness to take action.

Other vulnerable groups, such as those with limited English proficiency, individuals with disabilities, and those with limited access to technology or social media, may also face communication barriers that affect their ability to receive and understand wildfire smoke information.

Furthermore, the complexity of the information itself can be a barrier, as it may re

In [41]:
response = query_engine.query("How can hospitals, neonatal intensive care units, and nursing homes prepare for wildfire smoke events or evacuations?")
print(str(response))

To prepare for wildfire smoke events or evacuations, hospitals, neonatal intensive care units, and nursing homes should prioritize the health and safety of their patients and staff. This can be achieved by:

1. Developing emergency plans and protocols that account for wildfire smoke events and evacuations, including procedures for relocating patients and staff to safe areas.
2. Maintaining a supply of high-quality air filters, such as those with a MERV rating of 13 or higher, to ensure that indoor air quality remains safe for patients and staff.
3. Conducting regular drills and training exercises to ensure that staff are prepared to respond quickly and effectively in the event of a wildfire smoke event or evacuation.
4. Collaborating with local health authorities and emergency management officials to stay informed about wildfire smoke events and to receive timely updates on air quality and evacuation instructions.
5. Ensuring that patients and staff have access to personal protective e

In [42]:
response = query_engine.query("How does wildfire smoke complicate childcare, work, and wellbeing for farmworker families?")
print(str(response))

Wildfire smoke can pose significant challenges for farmworker families, particularly those with young children. The smoke can exacerbate respiratory issues and other health problems, making it difficult for parents to care for their children. 

The stress of dealing with wildfire smoke can also impact a family's ability to work, as parents may need to take time off to care for their children or to deal with their own health issues. This can be particularly challenging for farmworker families, who often rely on their income from farming to support their families.

Furthermore, the emotional toll of living through a wildfire event can be significant, and the ongoing stress of dealing with wildfire smoke can affect a family's overall wellbeing. This can lead to anxiety, depression, and other mental health issues, which can further complicate childcare, work, and overall family dynamics.


In [43]:
response = query_engine.query("How can social vulnerability and public risk perceptions be incorporated into wildfire preparedness and planning?")
print(str(response))

To effectively incorporate social vulnerability and public risk perceptions into wildfire preparedness and planning, it's essential to consider the diverse needs and concerns of various communities. This can be achieved by engaging in inclusive and participatory processes that involve local stakeholders, including residents, community leaders, and emergency management officials.

One approach is to conduct thorough risk assessments that take into account the social, economic, and environmental factors that contribute to wildfire risk. This can help identify areas of high vulnerability and inform targeted mitigation strategies.

Public risk perceptions can also be influenced by factors such as trust in emergency management agencies, access to information, and past experiences with wildfires. By understanding these perceptions, planners can develop more effective communication strategies and education programs that address community concerns and promote risk awareness.

Additionally, inc

In [44]:
response = query_engine.query("What should public health agencies consider when using prescribed fire as a wildfire mitigation strategy?")
print(str(response))

When utilizing prescribed fire as a wildfire mitigation strategy, public health agencies should consider the potential health impacts on individuals exposed to smoke from these fires. This includes assessing the air quality and monitoring for pollutants such as particulate matter, carbon monoxide, and volatile organic compounds. They should also be aware of the vulnerable populations, such as the elderly, young children, and those with pre-existing respiratory conditions, who may be more susceptible to the adverse effects of smoke exposure.


In [45]:
response = query_engine.query("How can citizen science, mobile apps, social media, and public messaging improve wildfire smoke awareness and protective action?")
print(str(response))

**Rewrite**

Citizen science initiatives can empower individuals to contribute to wildfire smoke monitoring and research, providing valuable data and insights. Mobile apps can offer real-time air quality information, allowing people to make informed decisions about their health and activities. Social media platforms can be leveraged to disseminate public health messages, raise awareness about wildfire smoke risks, and facilitate engagement with the public. Public messaging campaigns can utilize evidence-based messages, such as shocking and humorous content, to generate impressions and engagement, while information-based messages can be shared most to educate the public. By combining these approaches, it's possible to improve wildfire smoke awareness and encourage protective actions among the public, particularly in regions where social media is an inexpensive and effective method for delivering public health messages. Studies have shown that social media campaigns can be effective in c